# Document Preparation - Flat Pipeline

Demonstrates the end-to-end flat path:

```
PdfParser / DocxParser (mode="flat")
        ↓  Document.pages
FixedChunker / SentenceChunker / RecursiveChunker (string path)
        ↓  List[Chunk]
```

The flat path treats every document as a sequence of plain text pages.
It is fast, simple, and compatible with all existing chunkers.

## Imports

In [ ]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.chunker.fixed import FixedChunker
from cleave.chunker.recursive import RecursiveChunker
from cleave.chunker.sentence import SentenceChunker
from cleave.parsers.office.docx import DocxParser
from cleave.parsers.office.pdf import PdfParser
from cleave.schemas import ChunkParams, ChunkUnit

## Fixtures

In [ ]:
FIXTURES = pathlib.Path.cwd().parent.parent.parent / "tests" / "fixtures"
PDF_PATH  = str(FIXTURES / "sample.pdf")
DOCX_PATH = str(FIXTURES / "sample.docx")
print("PDF :", PDF_PATH)
print("DOCX:", DOCX_PATH)

## Parse (flat mode)

In [ ]:
pdf_doc  = PdfParser(PDF_PATH,   mode="flat").parse()
docx_doc = DocxParser(DOCX_PATH, mode="flat").parse()

print(f"PDF  — pages: {pdf_doc.total_pages}  chars: {len(pdf_doc.full_text)}")
print(f"DOCX — pages: {docx_doc.total_pages}  chars: {len(docx_doc.full_text)}")

## FixedChunker on PDF

In [ ]:
CHUNK_SIZE    = 200
CHUNK_OVERLAP = 20

fixed_chunks = FixedChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)).chunk(pdf_doc)

print(f"chunk_size={CHUNK_SIZE}  overlap={CHUNK_OVERLAP}  total={len(fixed_chunks)}\n")
for c in fixed_chunks:
    print(f"[{c.index}] page={c.page_number}  chars {c.char_start:>4}–{c.char_end:<4}  tokens={c.token_count:>3}  │  {c.text[:60]!r}")

## SentenceChunker on DOCX

In [ ]:
sent_chunks = SentenceChunker(ChunkParams(chunk_size=250, chunk_overlap=25)).chunk(docx_doc)

print(f"Total chunks: {len(sent_chunks)}\n")
for c in sent_chunks:
    print(f"[{c.index}] tokens={c.token_count:>3}  │  {c.text[:80]!r}")

## RecursiveChunker on PDF (string path)

In [ ]:
# RecursiveChunker on a flat document uses the delimiter-hierarchy string path.
# It prefers to split at paragraph boundaries (\n\n) before falling back to words.
rec_chunks = RecursiveChunker(ChunkParams(chunk_size=200, chunk_overlap=20)).chunk(pdf_doc)

print(f"Total chunks: {len(rec_chunks)}\n")
for c in rec_chunks:
    print(f"[{c.index}] page={c.page_number}  tokens={c.token_count:>3}  │  {c.text[:70]!r}")

## Chunker comparison

In [ ]:
print(f"{'Chunker':<18}  {'Chunks':>6}  {'Avg chars':>9}  {'Avg tokens':>10}")
print("-" * 48)
for label, chunks in [("FixedChunker", fixed_chunks), ("RecursiveChunker", rec_chunks)]:
    avg_chars  = sum(len(c.text) for c in chunks) / len(chunks)
    avg_tokens = sum(c.token_count for c in chunks) / len(chunks)
    print(f"{label:<18}  {len(chunks):>6}  {avg_chars:>9.1f}  {avg_tokens:>10.1f}")